## Implementación de algoritmo de segmentación circular y anotación de vídeo.

**Autores:** Lucia Fuentes González (210229), Tania Mobasser Aslfakouri (220299), Miriam Bernat Jiménez (210162)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Rutas de archivos
VIDEO_PATH = r"../videos/video_circulo.mp4"
VIDEO_OUT  = r"../videos/video_circulo_anotado.mp4"

# Parámetros del modelo pinhole  Z = (f · D_real) / d_px 
FOCAL_PX          = 360.0   # longitud focal estimada (px)
DIAMETRO_REAL_MM  = 65.0    # diámetro real del objeto circular (mm)

# Parámetros de segmentación 
MIN_CIRCULARIDAD  = 0.70    # 1.0 = círculo perfecto filtramos por encima de 0.70
MIN_AREA_PX       = 500     # área mínima en px^2 para descartar ruido

In [95]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"No se puede abrir el vídeo: {VIDEO_PATH}")

VIDEO_W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_FPS      = int(cap.get(cv2.CAP_PROP_FPS)) or 25
VIDEO_N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Vídeo: {VIDEO_W}x{VIDEO_H}  {VIDEO_FPS} fps  {VIDEO_N_FRAMES} frames")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, VIDEO_FPS, (VIDEO_W, VIDEO_H))

Z_est = []   # distancia estimada (mm) por frame, np.nan si no se detecta

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Preprocesado
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gray, (7, 7), 0)

    # 2. Detección de bordes (Canny)
    edges = cv2.Canny(blur, 50, 150)

    # 3. Extracción de contornos
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Seleccionar el contorno más circular con área suficiente
    best_cnt  = None
    best_circ = 0.0

    for cnt in contours:
        if len(cnt) < 5:
            continue

        area = cv2.contourArea(cnt)
        if area < MIN_AREA_PX:
            continue

        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        # 4. Filtro de circularidad
        circularidad = 4 * np.pi * area / (perimeter ** 2)
        if circularidad < MIN_CIRCULARIDAD:
            continue

        if circularidad > best_circ:
            best_circ = circularidad
            best_cnt  = cnt

    Z = np.nan
    if best_cnt is not None:
        # 5. Ajuste de elipse: estimación del diámetro en píxeles
        ellipse     = cv2.fitEllipse(best_cnt)
        (cx, cy)    = ellipse[0]               # centro de la elipse
        diameter_px = int(max(ellipse[1]))     # eje mayor como diámetro
        radius      = diameter_px // 2
        cx, cy      = int(cx), int(cy)

        # 6. Estimación de distancia: modelo pinhole
        #   Z [mm] = (focal [px] × D_real [mm]) / d_px [px]
        Z = (FOCAL_PX * DIAMETRO_REAL_MM) / diameter_px

        # 7. Visualización sobre el frame
        cv2.ellipse(frame, ellipse, (0, 255, 0), 2)
        cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

        cv2.putText(frame, f"Diametro: {diameter_px} px",
                    (20,  40), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
        cv2.putText(frame, f"Distancia: {Z/1000:.2f} m  ({Z:.0f} mm)",
                    (20,  80), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2)
        cv2.putText(frame, f"Circularidad: {best_circ:.2f}",
                    (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 150, 0), 2)
    else:
        cv2.putText(frame, "Circulo no detectado",
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2)

    Z_est.append(Z)
    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Procesados {frame_idx} frames.")
print(f"Vídeo anotado guardado en: {VIDEO_OUT}")


Vídeo: 480x480  29 fps  335 frames
Procesados 335 frames.
Vídeo anotado guardado en: ../videos/video_circulo_anotado.mp4
